# 04. Baseline Models

This notebook establishes baseline model performance on the minimally prepared dataset and defines the validation protocol for subsequent experiments.

In [1]:
from pathlib import Path
import sys

import pandas as pd

## Environment Setup

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = ARTIFACTS_DIR / "experiments"

EXPERIMENT_ID = "catboost_v0_raw_minimal_cv5"
RUN_TRAINING = False

Project root configured.
Processed directory configured.


In [3]:
from src.churn_ml.features import load_dataset
from src.churn_ml.modeling import (
    ExperimentResult,
    ExperimentConfig,
    load_experiment_results,
    save_experiment_result,
    utc_timestamp,
    create_stratified_cv,
    summarize_target,
    load_experiment,
)
from src.churn_ml.models.catboost import (
    CatBoostConfig,
    run_catboost_experiment,
)

c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Loading

In [4]:
dataset = load_dataset(
    version="v0_raw_minimal",
    processed_dir=PROCESSED_DIR,
)

print(f"Dataset version: {dataset.version}")
print(f"Train shape:     {dataset.X_train.shape}")
print(f"Target shape:    {dataset.y_train.shape}")
print(f"Test shape:      {dataset.X_test.shape}")

Dataset version: v0_raw_minimal
Train shape:     (10000, 205)
Target shape:    (10000,)
Test shape:      (2500, 205)


## Target Distribution

In [5]:
target_summary = summarize_target(dataset.y_train)

display(
    target_summary.style.format(
        {"proportion": "{:.2%}"}
    )
)

,class,count,proportion
0,0,8695,86.95%
1,1,1305,13.05%


## Validation Configuration

In [6]:
experiment_config = ExperimentConfig.default()

cv = create_stratified_cv(experiment_config)

print(f"Strategy:       {type(cv).__name__}")
print(f"Number of folds: {experiment_config.n_splits}")
print(f"Shuffle:         {experiment_config.shuffle}")
print(f"Random state:    {experiment_config.random_state}")
print(f"Primary metric:  {experiment_config.primary_metric}")
print(
    "Scoring metrics:",
    ", ".join(experiment_config.scoring_metrics),
)

Strategy:       StratifiedKFold
Number of folds: 5
Shuffle:         True
Random state:    42
Primary metric:  balanced_accuracy
Scoring metrics: balanced_accuracy, roc_auc, average_precision, log_loss


## CatBoost Baseline

In [7]:
catboost_config = CatBoostConfig.default()

if RUN_TRAINING:
    catboost_config = CatBoostConfig.default()

    catboost_experiment = run_catboost_experiment(
        dataset=dataset,
        config=experiment_config,
        model_config=catboost_config,
        experiment_id=EXPERIMENT_ID,
    )

    save_experiment_result(
        result=catboost_experiment.result,
        experiments_dir=EXPERIMENTS_DIR,
        fold_metrics=catboost_experiment.fold_metrics,
        oof_predictions=catboost_experiment.oof_predictions,
        test_predictions=catboost_experiment.test_predictions,
        overwrite=True,
    )
else:
    catboost_experiment = load_experiment(
        experiment_id=EXPERIMENT_ID,
        experiments_dir=EXPERIMENTS_DIR,
    )

## Experiment Results

In [8]:
display(catboost_experiment.fold_metrics)

display(
    pd.Series(
        catboost_experiment.result.metrics,
        name="value",
    ).to_frame()
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,decision_threshold,training_time_seconds,best_iteration
0,1,0.790897,0.970582,0.853891,0.149565,0.5,49.774234,609
1,2,0.770780,0.957099,0.809683,0.166769,0.5,64.126503,818
2,3,0.798078,0.960454,0.819719,0.164079,0.5,82.018493,473
3,4,0.822983,0.964896,0.841484,0.153321,0.5,90.873086,758
4,5,0.787159,0.958559,0.808492,0.166829,0.5,73.206437,611


,value
balanced_accuracy,0.793979
roc_auc,0.962040
average_precision,0.824716
log_loss,0.160113
decision_threshold,0.500000
optimized_threshold_oof,0.174000
optimized_balanced_accuracy_oof,0.894566
balanced_accuracy_mean,0.793979
balanced_accuracy_std,0.019054
roc_auc_mean,0.962318
